# Train neural network to calculate imaging systematic weights

## Inputs
- config file
- imaging attributes

## Outputs
- The best 5 NN model

In [6]:
import sys
sys.path.append("..")
from train import run_optuna_nn as run_optuna

from Weights import linear_weights, quadratic_weights, nn_weights

from pfstarget import cuts as Cuts

from pfsimaging import imaging as Im

import astropy.io.ascii as ascii
import astropy.io.fits as fits

import yaml

In [7]:
config = {}
path = "../configs/example_config.yaml"
with open(path, "r") as f:
    config = yaml.safe_load(f)

### Get target and imaging property data

In [ ]:
#load cosmology targets
Targets = loader.Target(config)
Targets.load_bsmask(config)

#load imaging attributes
property_file = os.path.join(config['Imaging']['base_dir'], config['Imaging']['imaging_dir'], 'all_property.fits') 
with fits.open(property_file) as hdu:
    Property = Table(hdu[1].data)

#Calculate the effective area corrected target density
pixel_data = Im.get_target_density(Targets, Property)

In [ ]:
# clean data

#remove anomaly
anomaly_pix = Im.anomaly()
cleaned_table = pixel_data[~np.isin(pixel_data['healpix'], anomaly_pix)]

# remove area too small, remove total too small, remove target density = 0, remove stellar density = 0, remove all nan

### Run Optuna

In [ ]:
keys = ['gseeing', 'rseeing', 'iseeing', 'zseeing', 'yseeing', 'star', 'g_depth', 'r_depth', 'i_depth', 'z_depth', 'y_depth', 'desi-csfd_extinction']
run_optuna_nn(pixel_data, keys, n_trials=200, top_k=5, run_name="test")

# Calculate imaging systematic weights

In [ ]:
# calculate the imaging systematic weights

# output is a astropy table including the imaging attributes, target density 
#and weights named ["lin_weight", "quad_weight", "nn_weight"] for each pixel

lin_weights = linear_weights(pixel_data, keys)
quad_weights = quadratic_weights(pixel_data, keys)
NN_weights = nn_weights(pixel_data, keys, config)

In [ ]:
save_dir = os.path.join(config['Imaging']['base_dir'], config['Imaging']['weights_dir'])

filename = os.path.join(save_dir, "linear_weights.fits")
lin_weights.write(filename, format='fits', overwrite=True)

filename = os.path.join(save_dir, "quadratic_weights.fits")
quad_weights.write(filename, format='fits', overwrite=True)

filename = os.path.join(save_dir, "nn_weights.fits")
NN_weights.write(filename, format='fits', overwrite=True)